### Base imports

In [25]:
# Setting up the python/jupyter environment
import pandas as pd
import ROOT
import matplotlib.pyplot as plt
import numpy as np
import json
import seaborn as sns
import sys

from IPython.display import Image
import tabulate

%matplotlib inline
plt.rcParams["figure.figsize"] = (15,8)
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

# import warnings
# warnings.filterwarnings("ignore", category=DeprecationWarning)


In [26]:
with open("../config/config.json") as f:
    config = json.load(f)
config

{'OUTPUT_DIR': 'root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/final/train_bdt/output',
 'AP_POST_PROCESS_DIR': 'root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/final/ap_post_process/validated'}

###  Data loading methods

In [35]:
#18: Xc_signal_Ypis_displaced_fromBs_fromDs
#19: Xc_signal_Ypis_displaced_fromB0_fromDp
#20: Xc_signal_Ypis_displaced_fromBp_fromD0
#24: signal

rdshad_dir = config["AP_POST_PROCESS_DIR"]

def mc_category_file(eventtype, category, polarity, sign="rs"):
    return  f"{rdshad_dir}/rds_categfiltered_2012_{eventtype}_{polarity}_{sign}_{str(category)}.root"

def mc_category_files(eventtype, category, sign="rs"):
    return [ mc_category_file(eventtype, category, p, sign=sign) for p in [ "magdown", "magup" ] ]

def mc_file(eventtype, polarity, sign="rs"):
    return  f"{rdshad_dir}/rds_categ_2012_{eventtype}_{polarity}_{sign}.root"

def mc_files(eventtype, sign="rs"):
    return [ mc_file(eventtype, p, sign=sign) for p in [ "magdown", "magup" ] ]

def mc_file_final(eventtype, polarity, sign="rs"):
    return  f"{rdshad_dir}/rds_final_2012_{eventtype}_{polarity}_{sign}.root"

def mc_files_final(eventtype, sign="rs"):
    return [ mc_file_final(eventtype, p, sign=sign) for p in [ "magdown", "magup" ] ]
    

def load_signal_rdf():
    """ Return a RDataFrame with the signal data, and appropriate cuts """
    rdf_signal = ROOT.RDataFrame("DecayTree",  mc_files_final("13563002"))
    print(f"Loaded {mc_files_final('13563002')}")
    # Extra cuts on signal, identical to the ones applied by Carmen
    # We make sure thatwe have true signak only, hence the cut on the Y TRUEID
    rdf_signal = rdf_signal.Filter("(BDT_3pi>(-0.0727) && BDT_Ds>(-0.0876) && BDT_Bs>(-0.0655) && (Xc_Selection>0) && (Xc_BKGCAT== 0 ) &&  Y_BKGCAT==50 && B_BKGCAT==50 && a\
bs(p1_fromXc_TRUEID)==321 && abs(p2_fromXc_TRUEID)==321 && abs(p3_fromXc_TRUEID)==211 && abs(p1_fromY_TRUEID)+abs(p2_fromY_TRUEID)+abs(p3_from\
Y_TRUEID)==3*211 )")
    # We also add the B_Y_SEP cut
    rdf_signal = rdf_signal.Filter("B_Y_SEP<-4.5 && BDT_Iso>0.03458 & q2_2>0 && abs(mN2v)<250e6")
    return rdf_signal

def load_incmc_category_rdf(category, inclmc_type):
    """ Return a RDataFrame with inclusive MC of a specific background category,
    for a specific event type  """
    rdf = ROOT.RDataFrame("DecayTree", mc_category_files( inclmc_type, category))
    return rdf.Filter("abs(mN2v)<250e6 & q2_2>0 ")

def load_incmc_from_inclMC_rdf(eventtype):
    """ Load the incmc, unfiltered. Exclude signal"""
    rdf1 = ROOT.RDataFrame("DecayTree",  mc_files(eventtype))
    return rdf1.Filter("abs(mN2v)<250e6 && q2_2>0  && category != 24")

def load_signal_from_inclMC_rdf():
    rdf = ROOT.RDataFrame("DecayTree",  mc_files("23903000"))
    rdf = rdf.Filter("category==24")
    rdf = rdf.Filter("(BDT_3pi>(-0.0727) && BDT_Ds>(-0.0876) && BDT_Bs>(-0.0655) && (Xc_Selection>0) && (Xc_BKGCAT== 0 ) &&  Y_BKGCAT==50 && B_BKGCAT==50 && abs(p1_fromXc_TRUEID)==321 && abs(p2_fromXc_TRUEID)==321 && abs(p3_fromXc_TRUEID)==211 && abs(p1_fromY_TRUEID)+abs(p2_fromY_TRUEID)+abs(p3_fromY_TRUEID)==3*211 )")
    rdf = rdf.Filter("B_Y_SEP<-4.5 && BDT_Iso>0.03458 & q2_2>0 && abs(mN2v)<250e6")
    return rdf

def load_category_rdf(category, inclmc_type):
    rdf = ROOT.RDataFrame("DecayTree", mc_category_files( inclmc_type, category))
    return rdf.Filter("abs(mN2v)<250e6")

def load_dataframes(inclmc_type):
    rdf_signal = load_signal_rdf()
    rdf_Xc_signal_Ypis_displaced_fromBs_fromDs = load_category_rdf(18, inclmc_type)
    rdf_Xc_signal_Ypis_displaced_fromB0_fromDp = load_category_rdf(19, inclmc_type)
    rdf_Xc_signal_Ypis_displaced_fromBp_fromD0 = load_category_rdf(20, inclmc_type)
    return (rdf_signal, rdf_Xc_signal_Ypis_displaced_fromBs_fromDs, rdf_Xc_signal_Ypis_displaced_fromB0_fromDp, rdf_Xc_signal_Ypis_displaced_fromBp_fromD0)




In [37]:
#Below are the variables used fir the clssification in TMVA
used_base_columns = [
    "Y_0_40_nc_mult",
    "Y_0_20_cc_mult",
    "Y_0_20_cc_PZ",
    "Y_0_30_nc_PZ",
    "Y_0_40_nc_PZ",
    "min_m2pi",
    "max_m2pi",
    "missing_mass_2",
    "B_M",
    # log(abs(PBsn))
    # log(abs(PBv/B_P))
    # log(abs(PBvn/B_P))
    # log(abs((PBsn-PBvn)/PBvn))
    # log(sqrt(abs(mDs2vn)))
    # log(Y_PE)
    "BDT_Iso",
    "B_pT_Bdir",
    "Y_BPVVDR",
    "missing_pY_mass",
    "Y_correctedMass", 
    "q2_2",
    "tauY_2",
    "B_BPVVDR", 
    "mN2v",
    "B_Y_SEP",
    "B_correctedMass",
     #"category",
     "eventIndex",
    "q2_2",
    "tauY_2",
]

cache_columns = used_base_columns + [
    "PBsn", "PBv", "B_P", "PBvn", "mDs2vn", "Y_PE" ]

cache_columns += [ f"p{i}_fromY_{c}" for i in range(1,4) for c in ["PE", "PX", "PY", "PZ"]]
cache_columns += [ f"p{i}_fromXc_{c}" for i in range(1,4) for c in ["PE", "PX", "PY", "PZ"]]

def add_derived_columns(df):
    df["log(abs(PBsn))"] = np.log(np.abs(df["PBsn"]))
    df["log(abs(PBv/B_P))"] = np.log(np.abs(df["PBv"] / df["B_P"]))
    df["log(abs(PBvn/B_P))"] = np.log(np.abs(df["PBvn"] / df["B_P"]))
    df["log(abs((PBsn-PBvn)/PBvn))"] = np.log(np.abs((df["PBsn"] - df["PBvn"]) / df["PBvn"]))
    df["log(sqrt(abs(mDs2vn)))"] = np.log(np.sqrt(np.abs(df["mDs2vn"])))
    df["log(Y_PE)"] = np.log(df["Y_PE"]) 
    df["diff_m2pi"] = df["max_m2pi"] - df["min_m2pi"]
    return df

def load_df(rdf, signal = 0):
    #print([ c for c in rdf.GetColumnNames() if 'p1_fromXc_P' in str(c) ])
    df = pd.DataFrame(rdf.Cache(cache_columns).AsNumpy())
    add_derived_columns(df) 
    df["signal"] = signal
    return df


# Useful groupby method
#------------------------------------------------------------------------------
def mygroupby(d, groupbycols):
    g = d.groupby(groupbycols).size().reset_index(name='count').sort_values([ 'count'], ascending=False).reset_index(drop=True)
    g["Percentage"] = g.apply(lambda row: 100 * row["count"]/d.shape[0], axis=1)
    g["cumulative %"] = g["Percentage"].cumsum(axis = 0)
    return g

In [43]:
def load_data(inclmc_type="23903000"):
    """ Returns a mad with the ROOT and Pandas
    dataframes for signal and categories 18, 19, 20 """
    # Creating the ROOT RDataFrames
    dataframes = load_dataframes(inclmc_type)
    (rdf_signal, rdf_disp_BsDs, rdf_disp_B0Dp, rdf_disp_BpD0) = dataframes
    # for rdf in dataframes:
    #      print(f"{rdf.Count().GetValue()}")
    root_datasets = {
        "rdf_signal": rdf_signal,
        "rdf_disp_BsDs": rdf_disp_BsDs,
        "rdf_disp_B0Dp": rdf_disp_B0Dp,
        "rdf_disp_BpD0": rdf_disp_BpD0,
    }

    # Creating Pandas dataframes based on the RDataFrames
    df_signal = load_df(rdf_signal, 1)
    df_disp_BsDs = load_df(rdf_disp_BsDs, 0)
    df_disp_B0Dp = load_df(rdf_disp_B0Dp, 0)
    df_disp_BpD0 = load_df(rdf_disp_BpD0, 0)
    pandas_datasets = {
        "df_signal": df_signal,
        "df_disp_BsDs": df_disp_BsDs,
        "df_disp_B0Dp": df_disp_B0Dp,
        "df_disp_BpD0": df_disp_BpD0,
    }

    for name, ds in pandas_datasets.items():
        print(f"Created {name}: {ds.shape}")
    
    return (root_datasets, pandas_datasets)

def load_signal_from_inclMC():
    """ The the signal sample i.e. category 24 from the
    inclusive MC, event type 23903000 """
    rdf  = load_signal_from_inclMC_rdf()
    df_signal = load_df(rdf, 1) 
    print(f"Created df_signal_23903000: {df_signal.shape}")
    return df_signal

def load_signal_all():
    """ Load the signal sample from both the dedicated event type
    and from the inclusive MC """
    
    # Beware we have mostly category 24, but also a few events of category 10 (diffVertex)
    rdf_signal = load_signal_rdf()
    df_signal = load_df(rdf_signal, 1)
    
    # These are all category 24
    rdf_signal_inclMC  = load_signal_from_inclMC_rdf()
    df_signal_incl_MC = load_df(rdf_signal_inclMC, 1)

    return pd.concat([df_signal, df_signal_incl_MC], axis=0)

def load_background_category(category):
    """ Load data for a specific background category from both
    inclusive MC samples
    """
    bkg23903000 = load_df(load_incmc_category_rdf(category, "23903000"), 0)
    bkg23903003 = load_df(load_incmc_category_rdf(category, "23903003"), 0)
    return pd.concat([bkg23903000, bkg23903003], axis=0)

def load_complete_df():
    """ Load a dataset with all categories 
    """
    df1 = load_df(load_incmc_from_inclMC_rdf("23903000"), 0)
    df2 = load_df(load_incmc_from_inclMC_rdf("23903003"), 0)
    df = pd.concat([df1, df2], axis=0)
    dfsig = load_signal_all()
    dfall = pd.concat([df, dfsig], axis=0)
    return dfall

print("rdf_signal = load_signal_rdf()")
print("df_signal = load_df(rdf_signal, 1)")

rdf_signal = load_signal_rdf()
df_signal = load_df(rdf_signal, 1)


## Checking the candidate count

In [44]:

# rdf1 = load_incmc_from_inclMC_rdf("23903000")
# rdf2 = load_incmc_from_inclMC_rdf("23903003")

# rdfs1 = load_signal_rdf()
# rdfs2 = load_signal_from_inclMC_rdf()

In [45]:
# sample = [[ "Inclusive MC", rdf1.Count().GetValue()],
#           [ "Double charm MC", rdf2.Count().GetValue()],
#           [ "Signal from Inclusive MC", rdfs2.Count().GetValue()],
#           [ "Signal MC", rdfs1.Count().GetValue()],
#          ]

In [46]:
# sdf = pd.DataFrame(sample, columns=["Desciption", "count"])
# print(sdf.to_latex())

## Categorization

In [47]:
# def mygroupby(d, groupbycols):
#     g = d.groupby(groupbycols).size().reset_index(name='count').sort_values([ 'count'], ascending=False).reset_index(drop=True)
#     g["Percentage"] = g.apply(lambda row: 100 * row["count"]/d.shape[0], axis=1)
#     g["cumulative %"] = g["Percentage"].cumsum(axis = 0)
#     return g

In [48]:
# df = load_complete_df()

## Checking the data

In [49]:
# rdf = load_incmc_from_inclMC_rdf("23903000")
# print(rdf.GetColumnNames())
# df=load_complete_df()
# mygroupby(df, 'category')
# mygroupby(df, 'eventIndex')

In [50]:
# (root_datasets, pandas_datasets) = load_data(inclmc_type="23903000")

In [51]:
 #(root_datasets, pandas_datasets) = load_data(inclmc_type="23903003")

## Checking the candidate count per category

In [52]:
# rdf = ROOT.RDataFrame("DecayTree", mc_files("23903000"))
# rdf = rdf.Filter("(BDT_3pi>(-0.0727) && BDT_Ds>(-0.0876) && BDT_Bs>(-0.0655) && (Xc_Selection>0) && (Xc_BKGCAT== 0 ) &&  Y_BKGCAT==50 && B_BKGCAT==50 && a\
# bs(p1_fromXc_TRUEID)==321 && abs(p2_fromXc_TRUEID)==321 && abs(p3_fromXc_TRUEID)==211 && abs(p1_fromY_TRUEID)+abs(p2_fromY_TRUEID)+abs(p3_from\
# Y_TRUEID)==3*211 )")
# rdf = rdf.Filter("B_Y_SEP<-4.5 && BDT_Iso>0.03458 & q2_2>0 && abs(mN2v)<250e6")
# mygroupby( pd.DataFrame(rdf.Cache("category").AsNumpy()), "category")

In [53]:
# rdf = ROOT.RDataFrame("DecayTree", mc_files("23903003"))
# mygroupby( pd.DataFrame(rdf.Cache("category").AsNumpy()), "category")

In [54]:
# df_signal = load_signal_all()
# df_background_18 = load_background_category(18)
# df_background_19 = load_background_category(19)
# df_background_20 = load_background_category(20)

In [55]:
# print(f"signal: {df_signal.shape[0]}")
# print(f"18: {df_background_18.shape[0]}")
# print(f"19: {df_background_19.shape[0]}")
# print(f"20: {df_background_20.shape[0]}")

In [56]:
# mygroupby(df_signal, 'category')

## Background categories

In [14]:
 categories = {
        "0": "Xc_background",
        "1": "Xc_signal_Ypis_diffAncestorYXc",
        "2": "Xc_signal_Ypis_B_vertex_fromBs",
        "3": "Xc_signal_Ypis_B_vertex_fromOtherB",
        "4": "Xc_signal_Ypis_B_vertex_fromHc",
        "5": "Xc_signal_Ypis_B_vertex_fromNone",
        "6": "Xc_signal_Ypis_nomatch_Prompt",
        "7": "Xc_signal_Ypis_nomatch_doubleCharm",
        "8": "Xc_signal_Ypis_nomatch_charmStrange",
        "9": "Xc_signal_Ypis_nomatch_Other",
        "10": "Xc_signal_Ypis_diffVertex_signal",
        "11": "Xc_signal_Ypis_diffVertex_tauFromB",
        "12": "Xc_signal_Ypis_diffVertex_normlike",
        "13": "Xc_signal_Ypis_diffVertex_doubleCharm",
        "14": "Xc_signal_Ypis_diffVertex_doubleCharm_OneFromB",
        "15": "Xc_signal_Ypis_diffVertex_doubleCharm_TwoFromB",
        "16": "Xc_signal_Ypis_diffVertex_CharmStrange",
        "17": "Xc_signal_Ypis_diffVertex_SomeFromPV",
        "18": "Xc_signal_Ypis_displaced_fromBs_fromDs",
        "19": "Xc_signal_Ypis_displaced_fromB0_fromDp",
        "20": "Xc_signal_Ypis_displaced_fromBp_fromD0",
        "21": "Xc_signal_Ypis_displaced_fromLambdab_fromLambdac",
        "22": "Xc_signal_Ypis_displaced_fromBs_fromDp",
        "23": "Xc_signal_Ypis_displaced_fromBp_fromDp",
        "24": "Xc_signal_Ypis_displaced_fromBs_fromTau",
        "25": "Xc_signal_Ypis_displaced_fromB0_fromD0",
        "26": "Xc_signal_Ypis_displaced_fromB0_fromDs",
        "27": "Xc_signal_Ypis_displaced_fromBs_fromD0",
        "28": "Xc_signal_Ypis_displaced_fromBp_fromDs",
        "29": "Xc_signal_Ypis_displaced_fromBs_fromDs_fromTau",
        "30": "Xc_signal_Ypis_displaced_fromLambdab_fromDs",
        "31": "Xc_signal_Ypis_displaced_fromLambdab_fromDp",
        "32": "Xc_signal_Ypis_displaced_fromXic",
        "33": "Xc_signal_Ypis_displaced_fromBs",
        "34": "Xc_signal_Ypis_displaced_fromB0_fromLambdac",
        "35": "Xc_signal_Ypis_displaced_fromB0_fromDs_fromTau",
        "36": "Xc_signal_Ypis_displaced_fromLambdab_fromD0",
        "37": "Xc_signal_Ypis_displaced_fromB0_fromDp_fromTau",
        "38": "Xc_signal_Ypis_displaced_NA",
        "39": "Xc_signal_Ypis_displaced_fromBp_fromDs_fromTau",
        "40": "Xc_signal_Ypis_displaced_fromBp",
        "41": "Xc_signal_Ypis_displaced_fromLambdab_fromDs_fromTau",
        "42": "Xc_signal_Ypis_displaced_fromBp_fromLambdac",
        "43": "Xc_signal_Ypis_displaced_fromDs",
        "44": "Xc_signal_Ypis_displaced_fromBs_fromDp_fromTau",
        "45": "Xc_signal_Ypis_displaced_fromBp_fromDp_fromTau",
        "46": "Xc_signal_Ypis_displaced_fromBs_fromLambdac",
        "47": "Xc_signal_Ypis_displaced_fromLambdab",
        "48": "Xc_signal_Ypis_displaced_fromB0",
        "49": "Xc_signal_Ypis_displaced_fromLambdac",
        "-1": "NA"
}